# 🛣️ Lane Detection FINE-TUNE — Vietnam ADAS (v2)
## Tiếp tục train từ `best.pt` hiện tại để bổ sung lane giữa + vạch vàng đôi

**Mục tiêu**: Không train lại từ đầu — chỉ **fine-tune tiếp** model `best.pt` hiện tại để:
- Nhận diện thêm **lane giữa** (không chỉ viền ngoài)
- Phân biệt rõ **vạch vàng đôi, vạch vàng đứt, vạch trắng liền**
- Cover **càng đầy đủ lane càng tốt** (variable lanes)
- **Giữ nguyên** phần vẽ bbox 2 bên tốt của model hiện tại

**Model base**: `best.pt` hiện tại (đã train 11-class VN) → upload sẵn lên Drive trước khi chạy notebook này.
**Model train**: `yolo11n-seg` (Ultralytics) — nhẹ, real-time cho ADAS.

### Pipeline tổng quan
1. Mount Google Drive + tải `best.pt` hiện tại về Colab/Kaggle
2. Tải **Curated datasets** (subset chọn lọc) để bổ sung lane giữa + vạch vàng:
   - **RLMD** (Taiwan, 25 classes — VN-style)
   - **Maadaa Lane Line Segmentation** (chọn subset 10-15k ảnh cover 35 classes)
   - **Curated subset** từ dataset merge hiện tại (lọc ảnh có lane giữa, vạch vàng)
3. Map tất cả về **11 classes chuẩn VN hiện tại** (giữ nguyên để tương thích)
4. **Fine-tune YOLO11n-seg** từ `best.pt` với augmentation mạnh hơn cho lane phụ
5. Validate + visualize predictions
6. Lưu `best.pt` mới về Drive → copy về local để chạy

### Quy trình upload `best.pt` hiện tại lên Drive
```
/Users/.../ADAS/backend/ai-service/ai_models/lane_detection/weights/
   ├── lane_vn.yaml
   └── best.pt   <-- model hiện tại (upload file này)
```
Upload vào Google Drive: `MyDrive/ADAS_Lane_VN/weights/best.pt`

### 11 Classes (giữ nguyên từ notebook gốc)
| ID | Class | Mô tả | Màu (BGR) |
|---|---|---|---|
| 0 | `background` | Nền | (0,0,0) |
| 1 | `road` | Mặt đường | (128,128,128) |
| 2 | `lane_white_solid` | Vạch trắng liền | (255,255,255) |
| 3 | `lane_white_dashed` | Vạch trắng đứt | (200,200,255) |
| 4 | `lane_yellow_solid` | Vạch vàng liền | (0,255,255) |
| 5 | `lane_yellow_dashed` | Vạch vàng đứt | (0,200,200) |
| 6 | `lane_double_yellow` | Vạch vàng đôi | (0,150,150) |
| 7 | `stop_line` | Vạch dừng | (0,0,255) |
| 8 | `crosswalk` | Vạch qua đường | (255,0,255) |
| 9 | `arrow_straight` | Mũi tên thẳng | (0,255,0) |
| 10 | `arrow_turn` | Mũi tên rẽ | (255,128,0) |

---
**Hyperparameters chính** (so với notebook gốc):
- `lr0=0.001` (thấp hơn 10× vì fine-tune, không phải train từ đầu)
- `epochs=30-50` (ít hơn vì model đã học)
- `mosaic=1.0`, `mixup=0.15`, `copy_paste=0.5` (augmentation mạnh hơn)
- `close_mosaic=10` (tắt mosaic 10 epoch cuối để model hội tụ)
- `freeze=5` (freeze 5 layer đầu của backbone — giữ feature extraction đã học)

## 📂 PHẦN 1 — Kết nối Google Drive + setup môi trường

Tự động detect Kaggle hay Colab để setup phù hợp.

In [ ]:
import os, sys, time, json, shutil, pathlib, subprocess, importlib
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle').exists()
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ and not IS_KAGGLE
print(f'[ENV] Kaggle = {IS_KAGGLE} | Colab = {IS_COLAB}')

if IS_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        DRIVE_ROOT = Path('/content/drive/MyDrive/ADAS_Lane_VN')
    except Exception:
        print('[DRIVE] Mount fail, fallback to /content/ADAS_Lane_VN')
        DRIVE_ROOT = Path('/content/ADAS_Lane_VN')
else:
    DRIVE_ROOT = Path('/kaggle/working/ADAS_Lane_VN')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DIRS = {
    "weights":  DRIVE_ROOT / "weights",
    "datasets": DRIVE_ROOT / "datasets",
    "runs":     DRIVE_ROOT / "runs",
    "hf_cache": DRIVE_ROOT / "hf_cache",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print(f'[DIRS] {DRIVE_ROOT}')
for k, v in DIRS.items():
    print(f'  {k:10s} : {v}')

## 📦 PHẦN 2 — Cài đặt thư viện

Cài các package cần thiết cho fine-tune.

In [ ]:
import subprocess, sys, importlib

def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

needed = [
    'ultralytics>=8.4.0',
    'huggingface_hub==0.27.1',
    'datasets==3.2.0',
    'opencv-python-headless>=4.10',
    'albumentations>=1.4',
    'onnx>=1.17',
    'onnxruntime>=1.18',
    'pyyaml>=6.0',
    'kaggle',
]
try:
    pip_install(needed)
    print('[PIP] All installed OK')
except subprocess.CalledProcessError as e:
    print(f'[PIP] FAIL: {e}')
    raise

import ultralytics, huggingface_hub, datasets, cv2, albumentations, onnx, onnxruntime
print(f'\n[VERSIONS]')
print(f'  ultralytics         : {ultralytics.__version__}')
print(f'  huggingface_hub     : {huggingface_hub.__version__}')
print(f'  datasets            : {datasets.__version__}')
print(f'  opencv              : {cv2.__version__}')
print(f'  albumentations      : {albumentations.__version__}')

## 🔐 PHẦN 3 — Đăng nhập Hugging Face & Kaggle + kiểm tra GPU

In [ ]:
import os, json, re, torch, requests, subprocess
from huggingface_hub import login, whoami

print('[API] Setup nguồn dataset (HF + Kaggle)...\n')

# ============================================================
# Load secrets from .env file (token KHÔNG được commit vào git)
# ============================================================
try:
    from dotenv import load_dotenv
    import pathlib
    # .env nằm cùng cấp với thư mục notebooks (root project)
    dotenv_path = pathlib.Path('/Users/melaniepham/Documents/Viet/2026/xử lý ảnh/DA/ADAS-MOI/ADAS/.env')
    if dotenv_path.exists():
        load_dotenv(dotenv_path)
        print('[ENV] Loaded secrets from .env')
    else:
        print('[ENV] .env not found — reading from environment')
except ImportError:
    print('[ENV] python-dotenv not installed — reading from environment')

# ---- HuggingFace ----
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '')
try:
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    user = whoami(token=os.environ['HF_TOKEN'])
    print(f'[HF]      OK    : {user.get("name", "?")} ({user.get("fullname", "?")})')
except Exception as e:
    print(f'[HF]      FAIL  : {e}')

# ---- Kaggle ----
KAGGLE_JSON = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
if not KAGGLE_JSON.exists():
    KAGGLE_JSON.parent.mkdir(parents=True, exist_ok=True)
    if IS_COLAB:
        try:
            from google.colab import files
            uploaded = files.upload()
            kaggle_dict = list(uploaded.values())[0]
            KAGGLE_JSON.write_bytes(kaggle_dict)
            KAGGLE_JSON.chmod(0o600)
            print('[KAGGLE]  OK    : uploaded from Colab')
        except Exception as e:
            print(f'[KAGGLE]  SKIP  : {e}')
    else:
        os.environ['KAGGLE_USERNAME'] = os.environ.get('KAGGLE_USERNAME', '')
        os.environ['KAGGLE_KEY'] = os.environ.get('KAGGLE_KEY', '')
        print('[KAGGLE]  OK    : from environment / .env')
else:
    print('[KAGGLE]  OK    : ~/.kaggle/kaggle.json exists')

# ---- GPU check ----
print(f'\n[GPU] {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA"}')
if torch.cuda.is_available():
    try:
        free, total = torch.cuda.mem_get_info()
        print(f'      VRAM   : {total/1024**3:.1f} GB total / {free/1024**3:.1f} GB free')
    except Exception:
        print(f'      VRAM   : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
    print(f'      CUDA   : {torch.version.cuda}')

## ⬇️ PHẦN 4 — Tải `best.pt` hiện tại về Colab/Kaggle

Model base = `best.pt` mà bạn đã upload lên Drive/Kaggle dataset.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

WEIGHTS_DIR = DIRS['weights']
BEST_PT = WEIGHTS_DIR / 'best.pt'

print(f'[BEST_PT TARGET] {BEST_PT}')
print(f'  Exists: {BEST_PT.exists()}')
print(f'  Size  : {BEST_PT.stat().st_size / 1024**2:.2f} MB' if BEST_PT.exists() else '  (need to download)')

if not BEST_PT.exists() and Path('/kaggle/input').exists():
    for ds in Path('/kaggle/input').iterdir():
        for pt in ds.rglob('best.pt'):
            print(f'[KAGGLE] Found best.pt in {ds.name}, copying...')
            shutil.copy2(pt, BEST_PT)
            print(f'[KAGGLE] Copied to {BEST_PT}')
            break
        if BEST_PT.exists():
            break

if not BEST_PT.exists() and IS_COLAB:
    drive_best = Path('/content/drive/MyDrive/ADAS_Lane_VN/weights/best.pt')
    if drive_best.exists():
        shutil.copy2(drive_best, BEST_PT)
        print(f'[DRIVE] Copied from {drive_best}')

if not BEST_PT.exists():
    print('[WARN] best.pt không có. Fallback: tải yolo11n-seg pretrained (sẽ train từ đầu)')
    print('       Upload best.pt lên Drive/Kaggle Dataset trước khi chạy!')
    BEST_PT = WEIGHTS_DIR / 'yolo11n-seg.pt'
    try:
        hf_hub_download(
            repo_id='Ultralytics/yolo11',
            filename='yolo11n-seg.pt',
            local_dir=str(WEIGHTS_DIR),
        )
    except Exception:
        if not BEST_PT.exists():
            subprocess.run(
                ['wget', '-q', '-O', str(BEST_PT),
                 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n-seg.pt'],
                check=True,
            )

print(f'\n[FINAL] base model = {BEST_PT}')
print(f'         size      = {BEST_PT.stat().st_size / 1024**2:.2f} MB')

## 📚 PHẦN 5 — Tải Curated Datasets (bổ sung lane giữa + vạch vàng)

Chọn **3 dataset nhỏ, chất lượng cao** để fine-tune:
1. **RLMD** (Road Line Marking Dataset, Taiwan) — 2,137 ảnh, 25 classes VN-style
2. **Maadaa Lane Line** (chọn subset 10-15k ảnh) — 35 classes chi tiết
3. **Curated subset** từ dataset merge hiện tại (lọc ảnh có lane giữa + vạch vàng)

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil, subprocess

DATA_ROOT = DIRS['datasets']

# ============================================================
# 1) RLMD (Road Line Marking Dataset, Taiwan)
# ============================================================
print('[1/3] RLMD (Taiwan, 2,137 ảnh, 25 classes)')
try:
    rlmd_path = snapshot_download(
        repo_id='veetinator/Road_Line_Marking_Dataset',
        repo_type='dataset',
        local_dir=str(DATA_ROOT / 'rlmd_raw'),
    )
    print(f'  -> Downloaded: {rlmd_path}')
except Exception as e:
    print(f'  FAIL: {e}')
    print('  Try Kaggle fallback...')
    try:
        subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', 'veetinator/road-line-marking-dataset',
             '-p', str(DATA_ROOT / 'rlmd_raw'), '--unzip'],
            check=True,
        )
        print('  -> Kaggle fallback OK')
    except Exception as e2:
        print(f'  Kaggle fallback FAIL: {e2}')

# ============================================================
# 2) Maadaa Lane Line Segmentation (subset)
# ============================================================
print('\n[2/3] Maadaa Lane Line Segmentation (subset 15k)')
try:
    maadaa_path = snapshot_download(
        repo_id='maadaa/lane-line-segmentation',
        repo_type='dataset',
        local_dir=str(DATA_ROOT / 'maadaa_raw'),
        max_workers=4,
    )
    print(f'  -> Downloaded: {maadaa_path}')
except Exception as e:
    print(f'  FAIL: {e}')
    print('  Tip: tải thủ công từ https://maadaa.ai và upload lên Drive')
    print('       Sau đó copy vào /content/drive/MyDrive/ADAS_Lane_VN/datasets/maadaa_raw/')

# ============================================================
# 3) Curated subset từ dataset merge hiện tại (nếu có)
# ============================================================
print('\n[3/3] Curated subset từ dataset merge (optional)')
merged_old = DIRS['datasets'].parent / 'merged_lane_vn'
if not merged_old.exists():
    candidates = [
        Path('/content/drive/MyDrive/ADAS_Lane_VN/datasets/merged_lane_vn'),
        Path('/kaggle/input/adas-lane-vn/merged_lane_vn'),
    ]
    for c in candidates:
        if c.exists():
            merged_old = c
            break
if merged_old.exists():
    print(f'  Found old merged dataset: {merged_old}')
    print('  Will curate in PHẦN 6')
else:
    print('  Old merged dataset not found (skip curated subset)')

print('\n[STATUS] Raw datasets downloaded to:', DATA_ROOT)
for d in DATA_ROOT.iterdir():
    if d.is_dir():
        n_files = sum(1 for _ in d.rglob('*') if _.is_file())
        print(f'  {d.name:30s} : {n_files} files')

## 🛠️ PHẦN 6 — Chuẩn hoá & Map dataset về **11 classes chuẩn VN**

Mapping rules:
- RLMD 25 classes → 11 classes VN
- Maadaa 35 classes → 11 classes VN
- Curated subset: copy nguyên (đã ở format VN)

In [ ]:
import cv2, json, yaml, shutil, random, numpy as np
from pathlib import Path
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

VN11_NAMES = [
    'background',
    'road',
    'lane_white_solid',
    'lane_white_dashed',
    'lane_yellow_solid',
    'lane_yellow_dashed',
    'lane_double_yellow',
    'stop_line',
    'crosswalk',
    'arrow_straight',
    'arrow_turn',
]

# Mapping RLMD 25 classes -> 11 classes VN
RLMD_TO_VN11 = {
    0: 0, 1: 9, 2: 8, 3: 7, 4: 2, 5: 4, 6: 2, 7: 2, 8: 6,
    9: 3, 10: 5, 11: 10, 12: 9, 13: 10, 14: 10, 15: 10,
    16: 2, 17: 8, 18: 7, 19: 2, 20: 2, 21: 2, 22: 8, 23: 8, 24: 2,
}

MERGED = DIRS['datasets'] / 'merged_lane_vn_v2'
MERGED.mkdir(parents=True, exist_ok=True)
(MERGED / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(MERGED / 'images' / 'val').mkdir(parents=True, exist_ok=True)
(MERGED / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
(MERGED / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
print(f'[MERGE OUT] {MERGED}')
print(f'  11 classes: {VN11_NAMES}')

In [ ]:
import zipfile

def unzip_all(extract_to: Path):
    zips = list(extract_to.rglob('*.zip'))
    print(f'[UNZIP] Found {len(zips)} zip files')
    for zp in zips:
        try:
            print(f'  Extracting {zp.name} ({zp.stat().st_size/1e6:.1f} MB)...')
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall(extract_to)
            print(f'    -> Done')
            zp.unlink()
        except Exception as e:
            print(f'    -> FAIL: {e}')
    print(f'[UNZIP] Final structure:')
    for p in sorted(extract_to.iterdir()):
        if p.is_dir():
            sub = list(p.iterdir())[:5]
            print(f'  {p.name}/ (sub: {[s.name for s in sub]})')
        else:
            print(f'  {p.name} ({p.stat().st_size/1e6:.1f} MB)')

unzip_all(DATA_ROOT)

In [ ]:
# ============================================================
# Convert RLMD -> YOLO-seg: auto-detect color per contour
# ============================================================
import cv2
from pathlib import Path
from tqdm import tqdm
import random

candidates = [d for d in DATA_ROOT.iterdir() if d.is_dir() and d.name.upper().startswith('RLMD')]
RLMD_RAW = candidates[0] if candidates else None

def detect_class_from_color(mean_rgb):
    """Map RGB color to VN11 class ID based on color properties."""
    r, g, b = mean_rgb
    total = r + g + b
    if total < 80:
        return None  # too dark, skip

    # Detect color type
    is_yellow = (r > 100 and g > 100 and b < 150)
    is_white  = (r > 150 and g > 150 and b > 150)
    is_red    = (r > 150 and g < 80 and b < 80)
    is_blue   = (b > 100 and r < 80 and g < 100)
    is_orange = (r > 200 and g > 100 and b < 50)

    # Stop line (class 7) - red or orange horizontal line
    # Crosswalk (class 8) - blue or white zebra pattern
    # Arrow (class 9/10) - green or white arrows
    # Road (class 1) - gray

    if is_yellow:
        return 5  # default yellow -> dashed (will refine below)
    elif is_white:
        return 3  # default white -> dashed
    elif is_red or is_orange:
        return 7  # stop line
    elif is_blue:
        return 8  # crosswalk
    else:
        return 3  # default: white dashed


def refine_class(cnt, mean_rgb, base_cls):
    """Refine class based on shape (solid vs dashed vs double)."""
    x, y, cw, ch = cv2.boundingRect(cnt)
    area = cv2.contourArea(cnt)
    if area < 200:
        return None

    # aspect_ratio = width / height
    # Horizontal lines (stop lines, crosswalks) have high width, low height
    # Vertical/slanted lines (lane markings) have high height, low width
    h, w_img = mean_rgb.shape[:2] if len(mean_rgb.shape) == 3 else (480, 640)
    aspect = cw / max(ch, 1)

    r, g, b = mean_rgb[:3] if len(mean_rgb) >= 3 else (200, 200, 200)
    is_yellow = (r > 100 and g > 100 and b < 150)
    is_white  = (r > 150 and g > 150 and b > 150)
    is_very_wide = aspect > 3.0  # very wide = stop line, crosswalk

    if is_very_wide:
        if is_yellow or is_white:
            return 7  # stop_line
        else:
            return 8  # crosswalk

    # Lane markings: classify solid vs dashed vs double
    # Double yellow: two parallel yellow contours very close
    if is_yellow and aspect > 1.2:
        return 6  # lane_double_yellow

    if is_white:
        if aspect > 1.2:
            return 2  # lane_white_solid
        else:
            return 3  # lane_white_dashed

    if is_yellow:
        if aspect > 1.0:
            return 4  # lane_yellow_solid
        else:
            return 5  # lane_yellow_dashed

    return base_cls


if RLMD_RAW is None:
    print('[RLMD] No RLMD folder, skip')
else:
    img_dir = RLMD_RAW / 'images'
    lbl_dir = RLMD_RAW / 'labels'

    lbl_all = list(lbl_dir.rglob('*')) if lbl_dir.exists() else []
    lbl_png = [p for p in lbl_all if p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
    img_files = [p for p in img_dir.rglob('*') if p.suffix.lower() in ('.jpg','.png','.jpeg')] if img_dir.exists() else []

    print(f'[RLMD] Folder: {RLMD_RAW.name}')
    print(f'  images/: {len(img_files)}')
    print(f'  labels/: {len(lbl_png)}')

    if not img_files or not lbl_png:
        print('[RLMD] No data, skip')
    else:
        # Analyze mask colors to understand the dataset
        print('[RLMD] Analyzing mask colors (first 20)...')
        all_colors = {}
        for mp in lbl_png[:20]:
            m = cv2.imread(str(mp), cv2.IMREAD_COLOR)
            if m is None:
                continue
            m_rgb = cv2.cvtColor(m, cv2.COLOR_BGR2RGB)
            mask_nonblack = (m_rgb.sum(axis=2) > 30)
            if mask_nonblack.sum() == 0:
                continue
            colors = m_rgb[mask_nonblack].reshape(-1, 3)
            colors_q = (colors // 10) * 10
            unique, counts = np.unique(colors_q, axis=0, return_counts=True)
            for c, n in zip(unique, counts):
                key = tuple(c)
                all_colors[key] = all_colors.get(key, 0) + n

        top = sorted(all_colors.items(), key=lambda x: -x[1])[:15]
        print('[RLMD] Top colors (R,G,B) -> count:')
        for color, count in top:
            print(f'  {color}: {count}')

        # Build label lookup
        label_lookup = {}
        for mp in lbl_png:
            stem = mp.stem
            for suffix in ['_label', '_mask', '_seg', '_gt']:
                if stem.endswith(suffix):
                    stem = stem[:-len(suffix)]
                    break
            label_lookup[stem] = mp

        # Class distribution tracking
        from collections import Counter
        class_stats = Counter()

        converted = 0
        matched = 0
        for img_path in tqdm(img_files, desc='RLMD/convert'):
            lbl_path = label_lookup.get(img_path.stem)
            if lbl_path is None:
                continue
            matched += 1

            mask = cv2.imread(str(lbl_path), cv2.IMREAD_COLOR)
            if mask is None:
                continue
            mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
            h_img, w_img = mask_rgb.shape[:2]

            gray_mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
            _, binary = cv2.threshold(gray_mask, 30, 255, cv2.THRESH_BINARY)

            if cv2.countNonZero(binary) < 100:
                continue

            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            yolo_lines = []

            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area < 200:
                    continue

                # Sample mean color from contour region
                cnt_mask = np.zeros((h_img, w_img), dtype=np.uint8)
                cv2.drawContours(cnt_mask, [cnt], -1, 255, -1)
                mean_color = cv2.mean(mask_rgb, mask=cnt_mask)
                r, g, b = mean_color[:3]

                # Detect class from color
                base_cls = detect_class_from_color((r, g, b))
                if base_cls is None:
                    continue

                # Refine based on shape
                cls_id = refine_class(cnt, (r, g, b, h_img, w_img), base_cls)
                if cls_id is None:
                    continue

                class_stats[cls_id] += 1

                poly = cnt.reshape(-1, 2).astype(float)
                poly[:, 0] /= w_img
                poly[:, 1] /= h_img
                yolo_lines.append(f'{cls_id} ' + ' '.join(f'{x:.6f}' for x in poly.flatten()))

            if not yolo_lines:
                continue

            out_split = 'train' if random.random() < 0.8 else 'val'
            out_img = MERGED / 'images' / out_split / f'rlmd_{img_path.name}'
            out_lbl = MERGED / 'labels' / out_split / f'rlmd_{img_path.stem}.txt'
            shutil.copy2(img_path, out_img)
            out_lbl.write_text('\n'.join(yolo_lines))
            converted += 1

        print(f'\n[RLMD] Matched: {matched}/{len(img_files)} | Converted: {converted}')
        print('[RLMD] Class distribution:')
        for cls_id, cnt in sorted(class_stats.items()):
            name = VN11_NAMES[cls_id] if cls_id < len(VN11_NAMES) else f'class_{cls_id}'
            print(f'  {cls_id:2d} {name:25s}: {cnt:5d}')

In [ ]:
# ============================================================
# Convert Maadaa (color-coded mask PNG) -> YOLO-seg format
# FIXED: hardcoded color palette + nearest-color matching
# ============================================================
MAADAA_RAW = DATA_ROOT / 'maadaa_raw'
if not MAADAA_RAW.exists():
    print('[MAADAA] Raw dir not found, skip')
else:
    print('[MAADAA] Convert masks -> YOLO-seg')

    # Hardcoded Maadaa color palette -> VN11 class
    # Format: (R, G, B) -> vn11_class_id
    # Based on Maadaa's standard color coding
    MAADAA_PALETTE = [
        # (R, G, B), class_id, class_name
        ((255, 255, 255), 2, 'lane_white_solid'),
        ((200, 200, 200), 3, 'lane_white_dashed'),
        ((180, 180, 180), 2, 'lane_white_solid'),
        ((220, 220, 220), 3, 'lane_white_dashed'),
        ((100, 100, 100), 1, 'road'),
        ((128, 128, 128), 1, 'road'),
        ((0, 255, 255),   5, 'lane_yellow_dashed'),
        ((0, 200, 200),   5, 'lane_yellow_dashed'),
        ((0, 220, 220),   5, 'lane_yellow_dashed'),
        ((0, 180, 180),   4, 'lane_yellow_solid'),
        ((0, 150, 150),   6, 'lane_double_yellow'),
        ((0, 100, 100),   6, 'lane_double_yellow'),
        ((50, 50, 50),     0, 'background'),
        ((0, 0, 0),       0, 'background'),
        ((0, 0, 255),     7, 'stop_line'),
        ((255, 0, 0),     7, 'stop_line'),
        ((255, 0, 255),   8, 'crosswalk'),
        ((200, 0, 200),   8, 'crosswalk'),
        ((0, 255, 0),     9, 'arrow_straight'),
        ((100, 200, 0),   9, 'arrow_straight'),
        ((255, 128, 0),   10, 'arrow_turn'),
        ((255, 165, 0),   10, 'arrow_turn'),
    ]

    def nearest_color_class(pixel_rgb):
        """Find nearest VN11 class from palette using Euclidean distance."""
        r, g, b = pixel_rgb
        best_dist = float('inf')
        best_cls = None
        for (pr, pg, pb), cls_id, _ in MAADAA_PALETTE:
            dist = (r - pr)**2 + (g - pg)**2 + (b - pb)**2
            if dist < best_dist:
                best_dist = dist
                best_cls = cls_id
        # Threshold: if distance too large, treat as background
        if best_dist > 150**2:
            return None
        return best_cls

    # Scan masks to understand actual colors used
    print('  Analyzing Maadaa mask colors...')
    all_mask_colors = Counter()
    for mask_path in list((MAADAA_RAW).rglob('*.png'))[:20]:
        mask = cv2.imread(str(mask_path))
        if mask is None:
            continue
        mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        unique = np.unique(mask_rgb.reshape(-1, 3), axis=0)
        for c in unique:
            if c.sum() > 30:  # skip near-black
                quantized = tuple((c // 20) * 20)
                all_mask_colors[quantized] += 1

    print(f'  Found {len(all_mask_colors)} unique quantized colors in first 20 masks')
    for color, cnt in all_mask_colors.most_common(10):
        print(f'    {color}: {cnt}')

    from collections import Counter
    converted = 0
    maadaa_class_stats = Counter()

    for split in ['train', 'val']:
        img_dir = MAADAA_RAW / split / 'images'
        mask_dir = MAADAA_RAW / split / 'masks'
        if not (img_dir.exists() and mask_dir.exists()):
            continue

        for img_path in tqdm(list(img_dir.glob('*.jpg'))[:5000], desc=f'Maadaa/{split}'):
            mask_path = mask_dir / (img_path.stem + '.png')
            if not mask_path.exists():
                mask_path = mask_dir / (img_path.stem + '.jpg')
            if not mask_path.exists():
                continue

            mask = cv2.imread(str(mask_path), cv2.IMREAD_COLOR)
            if mask is None:
                continue
            mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
            h_img, w_img = mask_rgb.shape[:2]

            yolo_lines = []

            # Get unique colors in this mask
            unique_colors = np.unique(mask_rgb.reshape(-1, 3), axis=0)

            for color_tuple in unique_colors:
                color_rgb = tuple(int(c) for c in color_tuple)
                if sum(color_rgb) < 50:
                    continue  # skip background/black

                cls_id = nearest_color_class(color_rgb)
                if cls_id is None:
                    continue

                # Create binary mask for this color
                diff = np.abs(mask_rgb.astype(int) - np.array(color_rgb)).sum(axis=2)
                binary = (diff < 30).astype(np.uint8) * 255

                if cv2.countNonZero(binary) < 100:
                    continue

                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if cv2.contourArea(cnt) < 200:
                        continue

                    maadaa_class_stats[cls_id] += 1

                    polygon = cnt.reshape(-1, 2).astype(float)
                    polygon[:, 0] /= w_img
                    polygon[:, 1] /= h_img
                    yolo_lines.append(f'{cls_id} ' + ' '.join(f'{x:.6f}' for x in polygon.flatten()))

            if not yolo_lines:
                continue

            out_split = 'train' if split == 'train' else 'val'
            out_img = MERGED / 'images' / out_split / f'maadaa_{img_path.name}'
            out_lbl = MERGED / 'labels' / out_split / f'maadaa_{img_path.stem}.txt'
            shutil.copy2(img_path, out_img)
            out_lbl.write_text('\n'.join(yolo_lines))
            converted += 1

    print(f'\n  -> Maadaa converted: {converted} images')
    print('  Maadaa class distribution:')
    for cls_id, cnt in sorted(maadaa_class_stats.items()):
        name = VN11_NAMES[cls_id] if cls_id < len(VN11_NAMES) else f'class_{cls_id}'
        print(f'    {cls_id:2d} {name:25s}: {cnt:5d}')

In [ ]:
# ============================================================
# Curated subset từ dataset merge hiện tại (lọc ảnh có >=2 lanes)
# ============================================================
if merged_old.exists():
    print(f'[CURATE] Filter old merged dataset: {merged_old}')
    src_train = merged_old / 'images' / 'train'
    src_val = merged_old / 'images' / 'val'
    src_lbl_train = merged_old / 'labels' / 'train'
    src_lbl_val = merged_old / 'labels' / 'val'

    n_curated = 0
    for split, src_img_dir, src_lbl_dir in [
        ('train', src_train, src_lbl_train),
        ('val', src_val, src_lbl_val),
    ]:
        if not src_img_dir.exists():
            continue
        for img_path in tqdm(list(src_img_dir.glob('*.jpg')) + list(src_img_dir.glob('*.png')), desc=f'Curate/{split}'):
            lbl_path = src_lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            lines = lbl_path.read_text().strip().split('\n')
            n_lanes = sum(1 for ln in lines if ln and int(ln.split()[0]) in {2,3,4,5,6})
            if n_lanes >= 2:
                out_img = MERGED / 'images' / split / f'curated_{img_path.name}'
                out_lbl = MERGED / 'labels' / split / f'curated_{img_path.stem}.txt'
                shutil.copy2(img_path, out_img)
                shutil.copy2(lbl_path, out_lbl)
                n_curated += 1

    print(f'  -> Curated subset: {n_curated} images (filtered to have >= 2 lanes)')
else:
    print('[CURATE] Old merged not found, skip')

# Final count
print('\n[FINAL MERGED DATASET]')
for split in ['train', 'val']:
    n = len(list((MERGED / 'images' / split).glob('*')))
    print(f'  {split:6s} : {n} images')

## 📝 PHẦN 7 — Sinh file `lane_vn.yaml` (11 classes chuẩn VN)

In [ ]:
import yaml

lane_vn_yaml = {
    'path': str(MERGED.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 11,
    'names': {
        0: 'background',
        1: 'road',
        2: 'lane_white_solid',
        3: 'lane_white_dashed',
        4: 'lane_yellow_solid',
        5: 'lane_yellow_dashed',
        6: 'lane_double_yellow',
        7: 'stop_line',
        8: 'crosswalk',
        9: 'arrow_straight',
        10: 'arrow_turn',
    },
}

yaml_path = DIRS['weights'] / 'lane_vn_v2.yaml'
yaml_path.write_text(yaml.dump(lane_vn_yaml, sort_keys=False))
print(f'[YAML] {yaml_path}')
print(yaml.dump(lane_vn_yaml, sort_keys=False))

## 🧐 PHẦN 8 — Sanity-check dataset

In [ ]:
from collections import Counter
from pathlib import Path

class_counter = Counter()
n_images_with_lanes = 0

for lbl_path in (MERGED / 'labels' / 'train').glob('*.txt'):
    lines = lbl_path.read_text().strip().split('\n')
    if not lines or lines == ['']:
        continue
    has_lane = False
    for line in lines:
        if line.strip():
            cls_id = int(line.split()[0])
            class_counter[cls_id] += 1
            if cls_id in {2, 3, 4, 5, 6}:
                has_lane = True
    if has_lane:
        n_images_with_lanes += 1

print('[CLASS DISTRIBUTION]')
print(f'Total train images: {sum(1 for _ in (MERGED / "labels" / "train").glob("*.txt"))}')
print(f'Images with lanes : {n_images_with_lanes}\n')
print(f'{"ID":<4} {"Class":<25} {"Count":<10} {"%":<6}')
print('-' * 50)
total = sum(class_counter.values()) or 1
for cls_id in sorted(class_counter.keys()):
    name = VN11_NAMES[cls_id] if cls_id < len(VN11_NAMES) else f'class_{cls_id}'
    cnt = class_counter[cls_id]
    pct = cnt / total * 100
    bar = '█' * int(pct / 2)
    print(f'{cls_id:<4} {name:<25} {cnt:<10} {pct:>5.1f}% {bar}')

low_classes = [cls_id for cls_id, cnt in class_counter.items() if cnt / total < 0.05 and cls_id in {2,3,4,5,6}]
if low_classes:
    print(f'\n⚠️  CẢNH BÁO: {len(low_classes)} class lane marking có <5% data:')
    for c in low_classes:
        print(f'    - {VN11_NAMES[c]} ({class_counter[c]} instances)')
    print('    -> Cân nhắc thêm data cho các class này, hoặc dùng class weights cao hơn.')

## 🚀 PHẦN 9 — Fine-tune YOLO11n-seg từ `best.pt`

In [ ]:
from ultralytics import YOLO
import torch, time

MODEL_PT = str(BEST_PT)
RUN_NAME = f'lane_vn_finetune_{int(time.time())}'
PROJECT_DIR = DIRS['runs']

print(f'[FINE-TUNE] Loading model: {MODEL_PT}')
model = YOLO(MODEL_PT)

n_cls = len(model.model.names) if hasattr(model.model, 'names') else 11
print(f'  Base model classes : {n_cls}')
if n_cls != 11:
    print(f'  ⚠️  WARNING: Model có {n_cls} classes, dataset cần 11.')
    print(f'     Ultralytics sẽ tự động remap output layer theo nc=11 trong yaml.')

print(f'\n[TRAIN] Fine-tune config:')
print(f'  Project dir : {PROJECT_DIR}')
print(f'  Run name    : {RUN_NAME}')
print(f'  Epochs      : 40')
print(f'  Imgsz       : 640')
print(f'  Batch       : 16 (adjust if OOM)')
print(f'  Device      : 0 (GPU)')

results = model.train(
    data=str(yaml_path),
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    # ---- Fine-tune specific ----
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,
    freeze=5,
    # ---- Augmentation ----
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=8,
    translate=0.1,
    scale=0.4,
    shear=2,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.5,
    close_mosaic=10,
    # ---- Logging ----
    plots=True,
    save=True,
    save_period=10,
    val=True,
    patience=15,
    verbose=True,
)

BEST_PT_NEW = PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'
LAST_PT_NEW = PROJECT_DIR / RUN_NAME / 'weights' / 'last.pt'
print(f'\n[OK] Fine-tune done')
print(f'  New best.pt : {BEST_PT_NEW}')
print(f'  Size        : {BEST_PT_NEW.stat().st_size / 1024**2:.2f} MB' if BEST_PT_NEW.exists() else '')

## 📊 PHẦN 10 — Đánh giá chi tiết trên tập Val

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print(f'[EVAL] Loading {BEST_PT_NEW}')
best_model = YOLO(str(BEST_PT_NEW))

metrics = best_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    save_json=True,
    conf=0.25,
    iou=0.6,
)

print(f'\n[VAL METRICS]')
print(f'  mAP50 (box)   : {metrics.box.map50:.4f}')
print(f'  mAP50-95 (box): {metrics.box.map:.4f}')
print(f'  mAP50 (mask)  : {metrics.seg.map50:.4f}')
print(f'  mAP50-95 (mask): {metrics.seg.map:.4f}')
print(f'  Precision     : {metrics.box.mp:.4f}')
print(f'  Recall        : {metrics.box.mr:.4f}')

print(f'\n[COMPARE vs BASE]')
print(f'  Base: {BEST_PT.name} ({BEST_PT.stat().st_size / 1024**2:.2f} MB)')
print(f'  New : {BEST_PT_NEW.name} ({BEST_PT_NEW.stat().st_size / 1024**2:.2f} MB)')

In [ ]:
import torch, time, numpy as np
from pathlib import Path

best_model = YOLO(str(BEST_PT_NEW))
model_info = {
    'model_size_MB': BEST_PT_NEW.stat().st_size / 1024**2,
    'params_M': sum(p.numel() for p in best_model.model.parameters()) / 1e6,
    'gflops': best_model.info()[1] if hasattr(best_model, 'info') else 0,
}
print(f'[MODEL INFO]')
print(f'  Size  : {model_info["model_size_MB"]:.2f} MB')
print(f'  Params: {model_info["params_M"]:.2f}M')
if model_info['gflops']:
    print(f'  GFLOPs: {model_info["gflops"]:.2f}')

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
for _ in range(5):
    best_model.predict(dummy, verbose=False)
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.time()
for _ in range(50):
    best_model.predict(dummy, verbose=False)
torch.cuda.synchronize() if torch.cuda.is_available() else None
elapsed = time.time() - t0
fps = 50 / elapsed
latency_ms = elapsed / 50 * 1000
print(f'\n[LATENCY @ 640x640]')
print(f'  FPS       : {fps:.1f}')
print(f'  Latency   : {latency_ms:.1f} ms/frame')

## 📦 PHẦN 11 — Export model để deploy ADAS

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print(f'[EXPORT] Loading {BEST_PT_NEW}')
best_model = YOLO(str(BEST_PT_NEW))

print('[1/2] Export ONNX...')
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    half=False,
    simplify=True,
    opset=13,
    dynamic=False,
    device=0,
)
print(f'  ONNX: {onnx_path}')

print('\n[2/2] Export TorchScript...')
try:
    ts_path = best_model.export(
        format='torchscript',
        imgsz=640,
        device=0,
    )
    print(f'  TorchScript: {ts_path}')
except Exception as e:
    print(f'  TorchScript FAIL: {e}')

print('\n[OK] Export done')

## 🖼️ PHẦN 12 — Visualize kết quả trên ảnh test

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt

print(f'[VIS] Loading {BEST_PT_NEW}')
best_model = YOLO(str(BEST_PT_NEW))

val_imgs = list((MERGED / 'images' / 'val').glob('*.jpg'))[:5] + list((MERGED / 'images' / 'val').glob('*.png'))[:5]
if not val_imgs:
    print('No val images found')
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    for i, img_path in enumerate(val_imgs[:6]):
        img = cv2.imread(str(img_path))
        results = best_model.predict(img, conf=0.25, verbose=False)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        axes[i].imshow(annotated_rgb)
        axes[i].set_title(f'{img_path.name}\n{len(results[0].boxes)} detections', fontsize=10)
        axes[i].axis('off')
    for j in range(len(val_imgs), 6):
        axes[j].axis('off')
    plt.tight_layout()
    plt.savefig(DIRS['runs'] / 'val_predictions.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {DIRS["runs"] / "val_predictions.png"}')

## ✅ PHẦN 13 — Tổng kết & checklist deploy

**Workflow từ Colab/Kaggle về local**:
1. Download `best.pt` mới từ Drive/Kaggle output
2. Copy về local: `backend/ai-service/ai_models/lane_detection/weights/best.pt`
3. Clear cache trong app (bấm `C` trên Streamlit)
4. Test trên ảnh VN thật

**Kết quả mong đợi**:
- Lane giữa được nhận diện (không chỉ viền ngoài)
- Phân biệt rõ vạch vàng đôi (`lane_double_yellow`), vạch vàng đứt (`lane_yellow_dashed`)
- Vạch trắng liền/đứt chính xác hơn
- mAP50(mask) > 0.5 (target), mAP50-95(mask) > 0.3

In [ ]:
import json
from pathlib import Path

summary = {
    'base_model': str(BEST_PT),
    'new_model': str(BEST_PT_NEW),
    'dataset_path': str(MERGED),
    'yaml_path': str(yaml_path),
    'n_train_images': len(list((MERGED / 'images' / 'train').glob('*'))),
    'n_val_images': len(list((MERGED / 'images' / 'val').glob('*'))),
    'run_name': RUN_NAME,
    'model_size_MB': BEST_PT_NEW.stat().st_size / 1024**2 if BEST_PT_NEW.exists() else 0,
}

summary_path = DIRS['runs'] / 'finetune_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('[SUMMARY]')
print(json.dumps(summary, indent=2))
print(f'\n[CHECKLIST DEPLOY]')
print(f'  1. Download: {BEST_PT_NEW.name} từ Drive/Kaggle')
print(f'  2. Copy to local: backend/ai-service/ai_models/lane_detection/weights/best.pt')
print(f'  3. Bấm C trên Streamlit để clear cache')
print(f'  4. Test trên ảnh VN thật')